In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

endpoint = "your-url.trycloudflare.com"
SILVER_PATH = "silver/idfm/arrets/"
GOLD_PATH = "gold/idfm/arrets/"

In [0]:
import boto3
from botocore.client import Config
import pandas as pd
import io

# Configure boto3 to connect to MinIO
s3_client = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    verify=True
)

# Test connection by listing buckets
try:
    buckets = s3_client.list_buckets()
    print(f"Successfully connected to MinIO at {endpoint}")
    print(f"Buckets: {[b['Name'] for b in buckets['Buckets']]}")
except Exception as e:
    print(f"Failed to connect: {e}")

Successfully connected to MinIO at https://assess-player-the-prisoners.trycloudflare.com
Buckets: ['idfm-data']


In [0]:
response = s3_client.list_objects_v2(Bucket='idfm-data', Prefix=SILVER_PATH, MaxKeys=10)

if 'Contents' in response:
    print(f"\nFound {len(response['Contents'])} objects (showing first 10):")
    parquet_files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    
    if parquet_files:
        # Read the first parquet file as an example
        first_file = parquet_files[0]
        print(f"\nReading: {first_file}")
        
        obj = s3_client.get_object(Bucket='idfm-data', Key=first_file)
        parquet_data = obj['Body'].read()
        
        # Read parquet data into pandas DataFrame
        df_pandas = pd.read_parquet(io.BytesIO(parquet_data))
        
        # Convert to Spark DataFrame
        df_silver = spark.createDataFrame(df_pandas)

        print(df_silver.printSchema())
        
        print(f"\nDataFrame shape: {df_pandas.shape}")
        print(df_silver.show(5))
        
    else:
        print("No parquet files found in the specified path")
else:
    print(f"No objects found with prefix: {SILVER_PATH}")


Found 1 objects (showing first 10):

Reading: silver/idfm/arrets/_ingestion_date=2026-08-26/data.parquet
root
 |-- arrid: string (nullable = true)
 |-- arrversion: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- changed_at: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- town: string (nullable = true)
 |-- postal_region: string (nullable = true)
 |-- accessibility: string (nullable = true)
 |-- audible_signals: string (nullable = true)
 |-- visual_signs: string (nullable = true)
 |-- fare_zone: string (nullable = true)
 |-- x_epsg2154: long (nullable = true)
 |-- y_epsg2154: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- zda_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)

None

DataFrame shape: (10000, 18)
+------+---------------+-------------------+-------------------+--------------------+-----+---------

In [0]:
gold_df = (
    df_silver
    .groupBy("town")
    .agg(
        # Total number of stops
        F.countDistinct("arrid").alias("total_stops"),

        # Number of different stop types
        F.countDistinct("type").alias("number_of_stop_types"),

        # Accessibility
        F.sum(
            F.when(
                F.col("accessibility").isNotNull(),
                1
            ).otherwise(0)
        ).alias("stops_with_accessibility_info"),

        # Audible signals
        F.sum(
            F.when(
                F.col("audible_signals").isNotNull(),
                1
            ).otherwise(0)
        ).alias("stops_with_audible_signals_info"),

        # Visual signals
        F.sum(
            F.when(
                F.col("visual_signs").isNotNull(),
                1
            ).otherwise(0)
        ).alias("stops_with_visual_signs_info"),

        # Fare zones
        F.countDistinct("fare_zone").alias("number_of_fare_zones"),

        # Geographic coverage
        F.sum(
            F.when(
                F.col("latitude").isNotNull() &
                F.col("longitude").isNotNull(),
                1
            ).otherwise(0)
        ).alias("stops_with_coordinates"),

        # Last update
        F.max("changed_at").alias("last_stop_update"),
    )
)

In [0]:
gold_df = (
    gold_df
    .withColumn(
        "coordinate_coverage_pct",
        F.round(
            F.col("stops_with_coordinates")
            / F.col("total_stops") * 100,
            2
        )
    )
)

gold_df = (
    gold_df
    .withColumn(
        "accessibility_info_coverage_pct",
        F.round(
            F.col("stops_with_accessibility_info")
            / F.col("total_stops") * 100,
            2
        )
    )
)

In [0]:
gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp()
)

In [0]:
gold_df.printSchema()

gold_df.show(
    20,
    truncate=False
)

root
 |-- town: string (nullable = true)
 |-- total_stops: long (nullable = false)
 |-- number_of_stop_types: long (nullable = false)
 |-- stops_with_accessibility_info: long (nullable = true)
 |-- stops_with_audible_signals_info: long (nullable = true)
 |-- stops_with_visual_signs_info: long (nullable = true)
 |-- number_of_fare_zones: long (nullable = false)
 |-- stops_with_coordinates: long (nullable = true)
 |-- last_stop_update: timestamp (nullable = true)
 |-- coordinate_coverage_pct: double (nullable = true)
 |-- accessibility_info_coverage_pct: double (nullable = true)
 |-- _gold_processed_at: timestamp (nullable = false)

+-----------------------+-----------+--------------------+-----------------------------+-------------------------------+----------------------------+--------------------+----------------------+-------------------+-----------------------+-------------------------------+-------------------------+
|town                   |total_stops|number_of_stop_types|stops_w

In [0]:
gold_pandas = gold_df.toPandas()

import io
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(
    gold_pandas,
    preserve_index=False
)

buffer = io.BytesIO()

pq.write_table(
    table,
    buffer
)

buffer.seek(0)

print(
    f"Parquet size: "
    f"{buffer.getbuffer().nbytes / 1024:.2f} KB"
)


Parquet size: 36.46 KB


In [0]:
from datetime import datetime
# Upload to MinIO using boto3
ingestion_date = datetime.now().strftime('%Y-%m-%d')
object_key = f"gold/idfm/arrets/_ingestion_date={ingestion_date}/data.parquet"

try:
    s3_client.put_object(
        Bucket='idfm-data',
        Key=object_key,
        Body=buffer.getvalue()
    )
    print(f"Successfully uploaded to MinIO: {object_key}")
    print(f"Rows written: {len(gold_pandas)}")
except Exception as e:
    print(f"Failed to upload: {e}")

Successfully uploaded to MinIO: gold/idfm/arrets/_ingestion_date=2026-08-26/data.parquet
Rows written: 1138
